In [ ]:
import os 
import shutil
from tqdm import tqdm
from config import SEV_DIR,train_src,test_src,val_src,df_train,df_test,df_val

In [ ]:
if os.path.exists(SEV_DIR):
    shutil.rmtree(SEV_DIR)

for split in ['train', 'val', 'test']:
    for label in ['1', '2', '3', '4']: # We only care about these classes now
        os.makedirs(os.path.join(SEV_DIR, split, label), exist_ok=True)

In [ ]:
#Helper function to copy only DR images
def prepare_severity_data(df, source_dir, split_name):
    print(f"Processing {split_name}...")
    copied_count = 0
    # Filter only rows where diagnosis > 0
    df_dr = df[df['diagnosis'] > 0]
    
    for _, row in tqdm(df_dr.iterrows(), total=len(df_dr)):
        fname = row["id_code"] + ".png"
        label = str(row["diagnosis"]) # 1, 2, 3, or 4
        
        src_path = os.path.join(source_dir, fname)
        dst_path = os.path.join(SEV_DIR, split_name, label, fname)
        
        if os.path.exists(src_path):
            shutil.copy(src_path, dst_path)
            copied_count += 1
            
    print(f"Moved {copied_count} images to {split_name}.")

In [ ]:
#Run the copy process
prepare_severity_data(df_train, train_src, 'train')
prepare_severity_data(df_val, val_src, 'val')
prepare_severity_data(df_test, test_src, 'test')

In [ ]:
#Check the balance
def count_sev_images(path):
    counts = {}
    if not os.path.exists(path): return counts
    for cls in sorted(os.listdir(path)):
        cls_path = os.path.join(path, cls)
        if os.path.isdir(cls_path):
            counts[cls] = len(os.listdir(cls_path))
    return counts

print("\n--- Severity Dataset Distribution ---")
print("Train:", count_sev_images(os.path.join(SEV_DIR, "train")))
print("Val:  ", count_sev_images(os.path.join(SEV_DIR, "val")))
print("Test: ", count_sev_images(os.path.join(SEV_DIR, "test")))